In [1]:
print("Hello")

Hello


In [2]:
import subprocess

subprocess.run(["ping", "-n", "1", "8.8.8.8"])

CompletedProcess(args=['ping', '-n', '1', '8.8.8.8'], returncode=1)

In [3]:
import subprocess
import re
import csv
import time
import math
import statistics
from datetime import datetime
from pathlib import Path

# ==========================================
# ICMP TEST CONFIGURATION
# ==========================================

TARGET_IP = "8.8.8.8"
PING_COUNT = 1000
INTERVAL_SECONDS = 1
TIMEOUT_MS = 2000
PACKET_SIZE = 32

# Output directory
OUTPUT_DIR = Path.home() / "Desktop" / "ICMP_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

RAW_FILE = OUTPUT_DIR / f"icmp_raw_{timestamp}.csv"
SUMMARY_FILE = OUTPUT_DIR / f"icmp_summary_{timestamp}.csv"


# ==========================================
# PERCENTILE CALCULATION
# ==========================================

def percentile(values, percent):
    if not values:
        return None

    values = sorted(values)

    rank = (len(values) - 1) * percent / 100

    lower = math.floor(rank)
    upper = math.ceil(rank)

    result = values[lower] + (
        values[upper] - values[lower]
    ) * (rank - lower)

    return round(result, 2)


# ==========================================
# SEND ICMP PING
# ==========================================

def send_ping(target):
    command = [
        "ping",
        "-4",
        "-n", "1",
        "-w", str(TIMEOUT_MS),
        "-l", str(PACKET_SIZE),
        target
    ]

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=(TIMEOUT_MS / 1000) + 3
        )

        output = result.stdout

        # Handles time=25ms and time<1ms
        match = re.search(
            r"time\s*([=<])\s*(\d+)\s*ms",
            output,
            re.IGNORECASE
        )

        if result.returncode == 0 and match:
            rtt = float(match.group(2))

            if match.group(1) == "<":
                # Windows reports only an upper bound
                # for time<1ms; exact RTT is unknown.
                return "Success", None, "<1ms"

            return "Success", rtt, "Measured"

        return "Failed", None, "No reply"

    except subprocess.TimeoutExpired:
        return "Timeout", None, "Process timeout"

    except Exception as error:
        return "Error", None, str(error)


# ==========================================
# RUN ICMP TEST
# ==========================================

def main():

    results = []
    rtts = []

    transmitted = 0
    received = 0

    print("=" * 55)
    print("       WINDOWS ICMP LATENCY TEST")
    print("=" * 55)
    print(f"Destination : {TARGET_IP}")
    print(f"Packets     : {PING_COUNT}")
    print(f"Interval    : {INTERVAL_SECONDS} sec")
    print(f"Timeout     : {TIMEOUT_MS} ms")
    print("=" * 55)

    test_start = time.monotonic()

    for sequence in range(1, PING_COUNT + 1):

        probe_start = time.monotonic()
        timestamp_now = datetime.now().isoformat(
            timespec="milliseconds"
        )

        status, rtt, note = send_ping(TARGET_IP)

        transmitted += 1

        if status == "Success":
            received += 1

            if rtt is not None:
                rtts.append(rtt)

        results.append({
            "Timestamp": timestamp_now,
            "Sequence": sequence,
            "Destination": TARGET_IP,
            "Status": status,
            "RTT_ms": rtt,
            "Note": note
        })

        print(
            f"Seq={sequence} | "
            f"Status={status} | "
            f"RTT={rtt if rtt is not None else note}"
        )

        # Maintain approximately one probe per interval.
        # If a timeout exceeds the interval, the next
        # probe begins immediately.
        elapsed = time.monotonic() - probe_start
        remaining = INTERVAL_SECONDS - elapsed

        if remaining > 0 and sequence < PING_COUNT:
            time.sleep(remaining)

    duration = time.monotonic() - test_start

    lost = transmitted - received
    loss_percent = (lost / transmitted) * 100

    summary = {
        "Destination": TARGET_IP,
        "Transmitted": transmitted,
        "Received": received,
        "Lost": lost,
        "Packet_Loss_Percent": round(loss_percent, 4),
        "Duration_Seconds": round(duration, 2),
        "RTT_Samples": len(rtts),
        "Min_RTT_ms": min(rtts) if rtts else None,
        "Avg_RTT_ms": (
            round(statistics.mean(rtts), 2)
            if rtts else None
        ),
        "Max_RTT_ms": max(rtts) if rtts else None,
        "P10_ms": percentile(rtts, 10),
        "P50_ms": percentile(rtts, 50),
        "P95_ms": percentile(rtts, 95),
        "P99_ms": percentile(rtts, 99)
    }

    # Save raw ICMP measurements
    with open(RAW_FILE, "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=results[0].keys()
        )
        writer.writeheader()
        writer.writerows(results)

    # Save summary
    with open(SUMMARY_FILE, "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=summary.keys()
        )
        writer.writeheader()
        writer.writerow(summary)

    print("\n" + "=" * 55)
    print("             ICMP TEST SUMMARY")
    print("=" * 55)

    for key, value in summary.items():
        print(f"{key:25}: {value}")

    print("\nRaw results:", RAW_FILE)
    print("Summary:", SUMMARY_FILE)


if __name__ == "__main__":
    main()


       WINDOWS ICMP LATENCY TEST
Destination : 8.8.8.8
Packets     : 1000
Interval    : 1 sec
Timeout     : 2000 ms
Seq=1 | Status=Failed | RTT=No reply
Seq=2 | Status=Failed | RTT=No reply
Seq=3 | Status=Failed | RTT=No reply
Seq=4 | Status=Failed | RTT=No reply
Seq=5 | Status=Failed | RTT=No reply
Seq=6 | Status=Failed | RTT=No reply
Seq=7 | Status=Failed | RTT=No reply
Seq=8 | Status=Failed | RTT=No reply
Seq=9 | Status=Failed | RTT=No reply
Seq=10 | Status=Failed | RTT=No reply
Seq=11 | Status=Failed | RTT=No reply
Seq=12 | Status=Failed | RTT=No reply
Seq=13 | Status=Failed | RTT=No reply
Seq=14 | Status=Failed | RTT=No reply
Seq=15 | Status=Failed | RTT=No reply
Seq=16 | Status=Failed | RTT=No reply
Seq=17 | Status=Failed | RTT=No reply
Seq=18 | Status=Failed | RTT=No reply
Seq=19 | Status=Failed | RTT=No reply
Seq=20 | Status=Failed | RTT=No reply
Seq=21 | Status=Failed | RTT=No reply
Seq=22 | Status=Failed | RTT=No reply
Seq=23 | Status=Failed | RTT=No reply
Seq=24 | Status=Fai

Great! Your screenshot now shows the exact location of `icmp_test.py`.

Copy and paste this command directly into your VS Code PowerShell terminal:

```
python "C:\Users\v-yaalam\Downloads\vscode_Dell_17-1\1.Python Learning\Network Coding\icmp_test.py"
```

Alternatively, navigate to the script's folder first:

```
cd "C:\Users\v-yaalam\Downloads\vscode_Dell_17-1\1.Python Learning\Network Coding"
```

Then execute:

```
python icmp_test.py
```

### Expected output

If you saved the ICMP testing code from earlier, you should see something similar to:

```
=======================================================
       WINDOWS ICMP LATENCY TEST
=======================================================
Destination : 8.8.8.8
Packets     : 1000
Interval    : 1 sec
Timeout     : 2000 ms
=======================================================

Seq=1 | Status=Success | RTT=25.0
Seq=2 | Status=Success | RTT=28.0
Seq=3 | Status=Success | RTT=26.0
...
```

After all probes complete, the script should display the P50, P95, P99, packet loss, and average RTT, and save the results as CSV files.

With 1,000 probes at approximately one-second intervals, the test should take around 17 minutes, potentially longer if probes time out.

Run the first command now. If another error appears, share the terminal output and I'll help you fix it.

Your Python script is an ICMP network performance testing tool. It sends ICMP Echo Requests to a destination, measures Round Trip Time (RTT), calculates latency percentiles (P10, P50, P95, P99), measures packet loss, and exports the results into CSV files.

I'll explain it section by section, including what each Python function does, why it is used, and how it relates to networking.

# 1. Overall workflow of the script

Start Python script

Load modules and test configuration

Send ICMP Echo Request

Windows ping → 8.8.8.8

Receive Echo Reply or timeout

Extract RTT, status, and timestamp

Repeat 1,000 times

Collect successful RTTs and failures

Calculate statistics

Min, average, max, P10, P50, P95, P99, packet loss

Export CSV reports

Raw packet results + summary statistics

# 2. Importing Python libraries

Your code begins with:

```
import subprocessimport reimport csvimport timeimport mathimport statisticsfrom datetime import datetimefrom pathlib import Path
```

In Python, `import` loads functionality from an existing module.

| Module       | Purpose in your script                        |
| ------------ | --------------------------------------------- |
| `subprocess` | Executes the Windows `ping` command           |
| `re`         | Extracts RTT values using regular expressions |
| `csv`        | Saves measurements into CSV files             |
| `time`       | Controls intervals and measures test duration |
| `math`       | Calculates percentile positions               |
| `statistics` | Calculates average RTT                        |
| `datetime`   | Records timestamps                            |
| `pathlib`    | Creates and manages file paths                |

For example:

```
import subprocesssubprocess.run(["ping", "-n", "1", "8.8.8.8"])
```

This executes the same Windows command you would type manually:

```
ping -n 1 8.8.8.8
```

Important: Your script does not construct ICMP packets itself. It uses the Windows `ping.exe` utility, which handles sending and receiving ICMP messages.

# 3. ICMP test configuration

```
TARGET_IP = "8.8.8.8"PING_COUNT = 1000INTERVAL_SECONDS = 1TIMEOUT_MS = 2000PACKET_SIZE = 32
```

These are configuration variables.

| Variable           | Value     | Explanation                            |
| ------------------ | --------- | -------------------------------------- |
| `TARGET_IP`        | `8.8.8.8` | Destination IP                         |
| `PING_COUNT`       | 1000      | Number of ICMP Echo Requests           |
| `INTERVAL_SECONDS` | 1         | Intended interval between probe starts |
| `TIMEOUT_MS`       | 2000      | Wait up to 2 seconds for a reply       |
| `PACKET_SIZE`      | 32        | ICMP payload size in bytes             |

### What happens when you run this?

Your Windows computer sends an ICMP Echo Request to `8.8.8.8`.

Windows 11

ICMP Echo Request (Type 8)

ICMP Echo Reply (Type 0)

8.8.8.8

RTT = time from sending the Echo Request until receiving its Echo Reply

Suppose:

- Request sent at 10:00:00.000
- Reply received at 10:00:00.065

Then:

\\[ RTT=0.065\times1000=65\text{ ms} \\]

That RTT is recorded as one successful measurement.

# 4. Creating the output directory

```
OUTPUT_DIR = Path.home() / "Desktop" / "ICMP_Results"OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
```

Let's understand each part.

`Path.home()`

Returns your user home directory. On your Windows computer, this would normally be:

```
C:\Users\v-yaalam
```

`/ "Desktop" / "ICMP_Results"`

The `/` operator in `pathlib` joins directory components. It is not division.

The resulting path is typically:

```
C:\Users\v-yaalam\Desktop\ICMP_Results
```

`mkdir(parents=True, exist_ok=True)`

- `mkdir()` creates a directory.
- `parents=True` creates missing parent directories if necessary.
- `exist_ok=True` prevents an error if the directory already exists.

Note that if your Windows Desktop is redirected to OneDrive, `Path.home() / "Desktop"` might not be your actual Desktop location.

## Generating unique filenames

```
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
```

This generates a date/time string.

For example:

```
20261014_091530
```

Then:

```
RAW_FILE = OUTPUT_DIR / f"icmp_raw_{timestamp}.csv"SUMMARY_FILE = OUTPUT_DIR / f"icmp_summary_{timestamp}.csv"
```

Creates filenames such as:

```
icmp_raw_20261014_091530.csv
icmp_summary_20261014_091530.csv
```

The `f` before the string means f-string, which allows inserting variable values using `{}`.

# 5. Understanding the percentile function

This is one of the most important sections.

```
def percentile(values, percent):    if not values:        return None    values = sorted(values)    rank = (len(values) - 1) * percent / 100    lower = math.floor(rank)    upper = math.ceil(rank)    result = values[lower] + (        values[upper] - values[lower]    ) * (rank - lower)    return round(result, 2)
```

Let's break it down.

### Step A — Define the function

```
def percentile(values, percent):
```

`def` defines a reusable function.

It accepts two arguments:

- `values`: A collection of RTT measurements.
- `percent`: The percentile you want, such as 50, 95, or 99.

For example:

```
percentile([10, 20, 30, 40, 50], 50)
```

Returns:

```
30.0
```

### Step B — Check for empty measurements

```
if not values:    return None
```

Suppose all 1,000 ICMP probes fail.

Then:

```
values = []
```

Without this check, the script would attempt to calculate a percentile from an empty list and raise an error.

`None` means there is no available value.

### Step C — Sort RTT measurements

```
values = sorted(values)
```

Suppose the RTT measurements are:

```
values = [80, 30, 100, 50, 20]
```

After sorting:

```
values = [20, 30, 50, 80, 100]
```

Percentiles require ordered values.

### Step D — Calculate the percentile position

```
rank = (len(values) - 1) * percent / 100
```

Here:

- `len(values)` returns the number of RTT samples.
- `percent` is the desired percentile.
- `rank` is the zero-based position in the sorted data.

Suppose we have five samples:

```
values = [20, 30, 50, 80, 100]
```

For P50:

\\[ rank=(5-1)\times\frac{50}{100}=2 \\]

Python uses zero-based indexing:

| Index | RTT    |
| ----- | ------ |
| 0     | 20 ms  |
| 1     | 30 ms  |
| 2     | 50 ms  |
| 3     | 80 ms  |
| 4     | 100 ms |

Therefore:

```
values[2] = 50
```

P50 is 50 ms.

### Step E — Understand floor and ceil

```
lower = math.floor(rank)upper = math.ceil(rank)
```

`math.floor()` rounds downward to the nearest integer.

`math.ceil()` rounds upward to the nearest integer.

For example:

```
rank = 3.8math.floor(3.8)  # 3math.ceil(3.8)   # 4
```

Why do we need both?

Because the percentile may fall between two RTT measurements.

### Step F — Linear interpolation

```
result = values[lower] + (    values[upper] - values[lower]) * (rank - lower)
```

Let's calculate P95 for the same five samples.

\\[ rank=(5-1)\times0.95=3.8 \\]

Therefore:

```
lower = 3upper = 4
```

The corresponding RTTs are:

```
values[3] = 80values[4] = 100
```

Substituting:

\\[ P95=80+(100-80)\times0.8 \\]

\\[ \boxed{P95=96\text{ ms}} \\]

The script estimates P95 between 80 ms and 100 ms.

This is called linear interpolation. It is a common percentile calculation method, although some testing systems use different definitions.

Interactive percentile example

P50P95P99

Calculated P95

# 96.0 ms

Position (rank)

# 3.80

20

Index 0

30

Index 1

50

Index 2

80

Index 3

100

Index 4

Sorted RTT samples: 20, 30, 50, 80, 100 ms. Highlighted cells are used for interpolation.

Calculation: 80 + (100 − 80) × 0.80 = 96.0 ms

# 6. Understanding the `send_ping()` function

```
def send_ping(target):
```

This function is responsible for sending one ICMP request and interpreting its result.

## A. Constructing the Windows ping command

```
command = [    "ping",    "-4",    "-n", "1",    "-w", str(TIMEOUT_MS),    "-l", str(PACKET_SIZE),    target]
```

Python creates a list of command arguments.

For your configuration, this is equivalent to:

```
ping -4 -n 1 -w 2000 -l 32 8.8.8.8
```

| Option    | Explanation                   |
| --------- | ----------------------------- |
| `-4`      | Force IPv4                    |
| `-n 1`    | Send one Echo Request         |
| `-w 2000` | Reply timeout of 2000 ms      |
| `-l 32`   | Send 32 bytes of ICMP payload |
| `8.8.8.8` | Destination                   |

`str(TIMEOUT_MS)` converts the integer `2000` into the string `"2000"` because command-line arguments are passed as strings.

## B. Executing the command

```
result = subprocess.run(    command,    capture_output=True,    text=True,    timeout=(TIMEOUT_MS / 1000) + 3)
```

This is how Python interacts with the Windows operating system.

`subprocess.run()`

Starts an external process, waits for it to finish, and returns the result.

`capture_output=True`

Captures the command's output rather than printing it directly.

`text=True`

Returns the output as a Python string instead of bytes.

`timeout=(TIMEOUT_MS / 1000) + 3`

Sets a Python process timeout of five seconds:

\\[ 2000/1000+3=5\text{ seconds} \\]

There are two different timeouts here:

- Windows ping timeout: 2 seconds waiting for an ICMP reply.
- Python process timeout: 5 seconds waiting for the `ping.exe` process to finish.

## C. Extracting the RTT

```
output = result.stdout
```

`stdout` means standard output.

For example, Windows might return:

```
Reply from 8.8.8.8: bytes=32 time=25ms TTL=117
```

Now Python must extract `25`.

Your code uses:

```
match = re.search(    r"time\s*([=<])\s*(\d+)\s*ms",    output,    re.IGNORECASE)
```

This is a regular expression (regex).

| Regex part      | Meaning                                |
| --------------- | -------------------------------------- |
| `time`          | Find the word time                     |
| `\s*`           | Allow zero or more spaces              |
| `([=<])`        | Match `=` or `<`                       |
| `(\d+)`         | Capture one or more digits             |
| `ms`            | Match milliseconds                     |
| `re.IGNORECASE` | Ignore uppercase/lowercase differences |

For example:

```
time=25ms
```

Produces:

```
match.group(1)  # "="match.group(2)  # "25"
```

Then:

```
rtt = float(match.group(2))
```

Converts `"25"` to the number `25.0`.

## D. Handling successful replies

```
if result.returncode == 0 and match:
```

This checks that the ping command reported success and the expected RTT text was found.

Then:

```
return "Success", rtt, "Measured"
```

Returns three values:

```
status = "Success"rtt = 25.0note = "Measured"
```

This is called tuple unpacking when the values are assigned to separate variables.

### What happens for `time<1ms`?

Windows sometimes displays:

```
Reply from 192.168.1.1: bytes=32 time<1ms TTL=64
```

Your script returns:

```
return "Success", None, "<1ms"
```

It counts the reply as received, but excludes it from RTT statistics because the exact value is unknown.

That means your percentile calculations could be biased if many replies are under 1 ms.

## E. Handling failures

```
except subprocess.TimeoutExpired:    return "Timeout", None, "Process timeout"
```

If the ping process exceeds five seconds, this handles the exception without crashing the whole script.

```
except Exception as error:    return "Error", None, str(error)
```

This catches other errors, such as an inability to start the ping process.

A failed ping command returns:

```
return "Failed", None, "No reply"
```

One limitation: this status is fairly generic. It could mean an ICMP timeout, destination unreachable, or another ping failure. The script doesn't distinguish every ICMP error.

# 7. Understanding the main function

```
def main():
```

This function coordinates the entire test.

It initializes:

```
results = []rtts = []transmitted = 0received = 0
```

### Why two lists?

`results` stores every probe, including failures.

Example:

```
results = [    {"Sequence": 1, "Status": "Success", "RTT_ms": 25},    {"Sequence": 2, "Status": "Failed", "RTT_ms": None},    {"Sequence": 3, "Status": "Success", "RTT_ms": 30}]
```

`rtts` stores only successfully measured RTT values.

```
rtts = [25.0, 30.0]
```

Why?

Because a packet timeout is not a measured RTT. You shouldn't automatically assign a timeout a latency of 2,000 ms when calculating RTT percentiles.

Packet loss is measured separately.

# 8. Understanding the `for` loop

```
for sequence in range(1, PING_COUNT + 1):
```

With:

```
PING_COUNT = 1000
```

This generates:

```
1, 2, 3, 4, 5, ... 1000
```

Python's `range()` excludes the final boundary, which is why the code uses `PING_COUNT + 1`.

For each iteration:

```
probe_start = time.monotonic()
```

Records the start time for interval measurement.

```
timestamp_now = datetime.now().isoformat(    timespec="milliseconds")
```

Records the wall-clock timestamp, such as:

```
2026-10-14T09:15:30.125
```

Then:

```
status, rtt, note = send_ping(TARGET_IP)
```

Sends one ping.

### Updating counters

```
transmitted += 1
```

This is shorthand for:

```
transmitted = transmitted + 1
```

If the probe succeeds:

```
if status == "Success":    received += 1    if rtt is not None:        rtts.append(rtt)
```

`append()` adds the RTT to the list.

For example:

```
rtts = [25, 30]rtts.append(28)
```

Becomes:

```
[25, 30, 28]
```

# 9. Storing every ICMP result

```
results.append({    "Timestamp": timestamp_now,    "Sequence": sequence,    "Destination": TARGET_IP,    "Status": status,    "RTT_ms": rtt,    "Note": note})
```

The `{}` syntax creates a Python dictionary.

Each dictionary stores one probe.

Example:

```
```

In [4]:

import socket
import time
import csv
import math
import statistics
from datetime import datetime
from pathlib import Path

# ==========================================
# TCP TEST CONFIGURATION
# ==========================================

TARGET_IP = "8.8.8.8"
TARGET_PORT = 53

TEST_COUNT = 1000
INTERVAL_SECONDS = 1
TIMEOUT_SECONDS = 2

# ==========================================
# OUTPUT FILES
# ==========================================

OUTPUT_DIR = Path.home() / "Desktop" / "TCP_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

RAW_FILE = OUTPUT_DIR / f"tcp_raw_{timestamp}.csv"
SUMMARY_FILE = OUTPUT_DIR / f"tcp_summary_{timestamp}.csv"

# ==========================================
# PERCENTILE FUNCTION
# ==========================================

def percentile(values, percent):
    if not values:
        return None

    values = sorted(values)

    rank = (len(values) - 1) * percent / 100

    lower = math.floor(rank)
    upper = math.ceil(rank)

    result = values[lower] + (
        values[upper] - values[lower]
    ) * (rank - lower)

    return round(result, 2)


# ==========================================
# TCP CONNECTION TEST
# ==========================================

def tcp_ping(ip, port, timeout):

    sock = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    )

    sock.settimeout(timeout)

    start = time.perf_counter()

    try:
        sock.connect((ip, port))

        end = time.perf_counter()

        latency_ms = (end - start) * 1000

        return "Success", round(latency_ms, 3), "Connected"

    except socket.timeout:
        return "Timeout", None, "Connection timeout"

    except ConnectionRefusedError:
        return "Refused", None, "Connection refused"

    except OSError as error:
        return "Error", None, str(error)

    finally:
        sock.close()


# ==========================================
# MAIN TCP TEST
# ==========================================

def main():

    results = []
    latencies = []

    print("=" * 60)
    print("           WINDOWS TCP PING TEST")
    print("=" * 60)
    print(f"Destination : {TARGET_IP}")
    print(f"TCP Port    : {TARGET_PORT}")
    print(f"Test Count  : {TEST_COUNT}")
    print(f"Interval    : {INTERVAL_SECONDS} seconds")
    print(f"Timeout     : {TIMEOUT_SECONDS} seconds")
    print("=" * 60)

    test_start = time.monotonic()

    for sequence in range(1, TEST_COUNT + 1):

        probe_start = time.monotonic()

        timestamp_now = datetime.now().isoformat(
            timespec="milliseconds"
        )

        status, latency, note = tcp_ping(
            TARGET_IP,
            TARGET_PORT,
            TIMEOUT_SECONDS
        )

        results.append({
            "Timestamp": timestamp_now,
            "Sequence": sequence,
            "Destination": TARGET_IP,
            "Port": TARGET_PORT,
            "Status": status,
            "TCP_Connect_ms": latency,
            "Note": note
        })

        if status == "Success":
            latencies.append(latency)

        print(
            f"Seq={sequence} | "
            f"TCP={TARGET_PORT} | "
            f"Status={status} | "
            f"Connect={latency} ms"
        )

        elapsed = time.monotonic() - probe_start
        remaining = INTERVAL_SECONDS - elapsed

        if remaining > 0 and sequence < TEST_COUNT:
            time.sleep(remaining)

    duration = time.monotonic() - test_start

    # ==========================================
    # TCP STATISTICS
    # ==========================================

    transmitted = len(results)
    successful = len(latencies)
    failed = transmitted - successful

    failure_percent = (
        failed / transmitted * 100
        if transmitted else 0
    )

    timeout_count = sum(
        1 for r in results if r["Status"] == "Timeout"
    )

    refused_count = sum(
        1 for r in results if r["Status"] == "Refused"
    )

    error_count = sum(
        1 for r in results if r["Status"] == "Error"
    )

    # Consecutive failure statistics
    current_streak = 0
    max_failure_streak = 0

    for r in results:
        if r["Status"] != "Success":
            current_streak += 1
            max_failure_streak = max(
                max_failure_streak,
                current_streak
            )
        else:
            current_streak = 0

    summary = {
        "Destination": TARGET_IP,
        "Port": TARGET_PORT,
        "Attempts": transmitted,
        "Successful": successful,
        "Failed": failed,
        "Failure_Percent": round(failure_percent, 4),
        "Timeouts": timeout_count,
        "Refused": refused_count,
        "Errors": error_count,
        "Max_Consecutive_Failures": max_failure_streak,
        "Duration_Seconds": round(duration, 2),
        "Min_TCP_ms": min(latencies) if latencies else None,
        "Avg_TCP_ms": (
            round(statistics.mean(latencies), 2)
            if latencies else None
        ),
        "Max_TCP_ms": max(latencies) if latencies else None,
        "P10_ms": percentile(latencies, 10),
        "P50_ms": percentile(latencies, 50),
        "P95_ms": percentile(latencies, 95),
        "P99_ms": percentile(latencies, 99)
    }

    # ==========================================
    # EXPORT RAW RESULTS
    # ==========================================

    with open(RAW_FILE, "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=results[0].keys()
        )
        writer.writeheader()
        writer.writerows(results)

    # ==========================================
    # EXPORT SUMMARY
    # ==========================================

    with open(SUMMARY_FILE, "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=summary.keys()
        )
        writer.writeheader()
        writer.writerow(summary)

    # ==========================================
    # DISPLAY RESULTS
    # ==========================================

    print("\n" + "=" * 60)
    print("             TCP TEST SUMMARY")
    print("=" * 60)

    for key, value in summary.items():
        print(f"{key:30}: {value}")

    print("\nRaw CSV:", RAW_FILE)
    print("Summary CSV:", SUMMARY_FILE)


if __name__ == "__main__":
    main()


           WINDOWS TCP PING TEST
Destination : 8.8.8.8
TCP Port    : 53
Test Count  : 1000
Interval    : 1 seconds
Timeout     : 2 seconds
Seq=1 | TCP=53 | Status=Success | Connect=25.453 ms
Seq=2 | TCP=53 | Status=Success | Connect=10.765 ms
Seq=3 | TCP=53 | Status=Success | Connect=10.268 ms
Seq=4 | TCP=53 | Status=Success | Connect=9.284 ms
Seq=5 | TCP=53 | Status=Success | Connect=11.482 ms
Seq=6 | TCP=53 | Status=Success | Connect=9.864 ms
Seq=7 | TCP=53 | Status=Success | Connect=12.434 ms
Seq=8 | TCP=53 | Status=Success | Connect=8.873 ms
Seq=9 | TCP=53 | Status=Success | Connect=11.342 ms
Seq=10 | TCP=53 | Status=Success | Connect=9.65 ms
Seq=11 | TCP=53 | Status=Success | Connect=22.48 ms
Seq=12 | TCP=53 | Status=Success | Connect=26.143 ms
Seq=13 | TCP=53 | Status=Success | Connect=13.642 ms
Seq=14 | TCP=53 | Status=Success | Connect=9.191 ms
Seq=15 | TCP=53 | Status=Success | Connect=12.941 ms
Seq=16 | TCP=53 | Status=Success | Connect=11.175 ms
Seq=17 | TCP=53 | Status=Succ

In [5]:

import socket
import time
import csv
import math
import statistics
from datetime import datetime
from pathlib import Path

# ==========================================
# TCP TEST CONFIGURATION
# ==========================================

TARGET_IP = "8.8.8.8"
TARGET_PORT = 53

TEST_COUNT = 10
INTERVAL_SECONDS = 1
TIMEOUT_SECONDS = 2

# ==========================================
# OUTPUT FILES
# ==========================================

OUTPUT_DIR = Path.home() / "Desktop" / "TCP_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

RAW_FILE = OUTPUT_DIR / f"tcp_raw_{timestamp}.csv"
SUMMARY_FILE = OUTPUT_DIR / f"tcp_summary_{timestamp}.csv"

# ==========================================
# PERCENTILE FUNCTION
# ==========================================

def percentile(values, percent):
    if not values:
        return None

    values = sorted(values)

    rank = (len(values) - 1) * percent / 100

    lower = math.floor(rank)
    upper = math.ceil(rank)

    result = values[lower] + (
        values[upper] - values[lower]
    ) * (rank - lower)

    return round(result, 2)


# ==========================================
# TCP CONNECTION TEST
# ==========================================

def tcp_ping(ip, port, timeout):

    sock = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    )

    sock.settimeout(timeout)

    start = time.perf_counter()

    try:
        sock.connect((ip, port))

        end = time.perf_counter()

        latency_ms = (end - start) * 1000

        return "Success", round(latency_ms, 3), "Connected"

    except socket.timeout:
        return "Timeout", None, "Connection timeout"

    except ConnectionRefusedError:
        return "Refused", None, "Connection refused"

    except OSError as error:
        return "Error", None, str(error)

    finally:
        sock.close()


# ==========================================
# MAIN TCP TEST
# ==========================================

def main():

    results = []
    latencies = []

    print("=" * 60)
    print("           WINDOWS TCP PING TEST")
    print("=" * 60)
    print(f"Destination : {TARGET_IP}")
    print(f"TCP Port    : {TARGET_PORT}")
    print(f"Test Count  : {TEST_COUNT}")
    print(f"Interval    : {INTERVAL_SECONDS} seconds")
    print(f"Timeout     : {TIMEOUT_SECONDS} seconds")
    print("=" * 60)

    test_start = time.monotonic()

    for sequence in range(1, TEST_COUNT + 1):

        probe_start = time.monotonic()

        timestamp_now = datetime.now().isoformat(
            timespec="milliseconds"
        )

        status, latency, note = tcp_ping(
            TARGET_IP,
            TARGET_PORT,
            TIMEOUT_SECONDS
        )

        results.append({
            "Timestamp": timestamp_now,
            "Sequence": sequence,
            "Destination": TARGET_IP,
            "Port": TARGET_PORT,
            "Status": status,
            "TCP_Connect_ms": latency,
            "Note": note
        })

        if status == "Success":
            latencies.append(latency)

        print(
            f"Seq={sequence} | "
            f"TCP={TARGET_PORT} | "
            f"Status={status} | "
            f"Connect={latency} ms"
        )

        elapsed = time.monotonic() - probe_start
        remaining = INTERVAL_SECONDS - elapsed

        if remaining > 0 and sequence < TEST_COUNT:
            time.sleep(remaining)

    duration = time.monotonic() - test_start

    # ==========================================
    # TCP STATISTICS
    # ==========================================

    transmitted = len(results)
    successful = len(latencies)
    failed = transmitted - successful

    failure_percent = (
        failed / transmitted * 100
        if transmitted else 0
    )

    timeout_count = sum(
        1 for r in results if r["Status"] == "Timeout"
    )

    refused_count = sum(
        1 for r in results if r["Status"] == "Refused"
    )

    error_count = sum(
        1 for r in results if r["Status"] == "Error"
    )

    # Consecutive failure statistics
    current_streak = 0
    max_failure_streak = 0

    for r in results:
        if r["Status"] != "Success":
            current_streak += 1
            max_failure_streak = max(
                max_failure_streak,
                current_streak
            )
        else:
            current_streak = 0

    summary = {
        "Destination": TARGET_IP,
        "Port": TARGET_PORT,
        "Attempts": transmitted,
        "Successful": successful,
        "Failed": failed,
        "Failure_Percent": round(failure_percent, 4),
        "Timeouts": timeout_count,
        "Refused": refused_count,
        "Errors": error_count,
        "Max_Consecutive_Failures": max_failure_streak,
        "Duration_Seconds": round(duration, 2),
        "Min_TCP_ms": min(latencies) if latencies else None,
        "Avg_TCP_ms": (
            round(statistics.mean(latencies), 2)
            if latencies else None
        ),
        "Max_TCP_ms": max(latencies) if latencies else None,
        "P10_ms": percentile(latencies, 10),
        "P50_ms": percentile(latencies, 50),
        "P95_ms": percentile(latencies, 95),
        "P99_ms": percentile(latencies, 99)
    }

    # ==========================================
    # EXPORT RAW RESULTS
    # ==========================================

    with open(RAW_FILE, "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=results[0].keys()
        )
        writer.writeheader()
        writer.writerows(results)

    # ==========================================
    # EXPORT SUMMARY
    # ==========================================

    with open(SUMMARY_FILE, "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=summary.keys()
        )
        writer.writeheader()
        writer.writerow(summary)

    # ==========================================
    # DISPLAY RESULTS
    # ==========================================

    print("\n" + "=" * 60)
    print("             TCP TEST SUMMARY")
    print("=" * 60)

    for key, value in summary.items():
        print(f"{key:30}: {value}")

    print("\nRaw CSV:", RAW_FILE)
    print("Summary CSV:", SUMMARY_FILE)


if __name__ == "__main__":
    main()


           WINDOWS TCP PING TEST
Destination : 8.8.8.8
TCP Port    : 53
Test Count  : 10
Interval    : 1 seconds
Timeout     : 2 seconds
Seq=1 | TCP=53 | Status=Success | Connect=11.172 ms
Seq=2 | TCP=53 | Status=Success | Connect=9.606 ms
Seq=3 | TCP=53 | Status=Success | Connect=8.838 ms
Seq=4 | TCP=53 | Status=Success | Connect=10.258 ms
Seq=5 | TCP=53 | Status=Success | Connect=10.08 ms
Seq=6 | TCP=53 | Status=Success | Connect=9.04 ms
Seq=7 | TCP=53 | Status=Success | Connect=10.62 ms
Seq=8 | TCP=53 | Status=Success | Connect=16.093 ms
Seq=9 | TCP=53 | Status=Success | Connect=27.683 ms
Seq=10 | TCP=53 | Status=Success | Connect=11.306 ms

             TCP TEST SUMMARY
Destination                   : 8.8.8.8
Port                          : 53
Attempts                      : 10
Successful                    : 10
Failed                        : 0
Failure_Percent               : 0.0
Timeouts                      : 0
Refused                       : 0
Errors                        : 0


In [6]:

import socket
import time
import csv
import math
import statistics
import ipaddress
from datetime import datetime
from pathlib import Path

# ==========================================
# USER INPUT CONFIGURATION
# ==========================================

print("=" * 60)
print("        TCP NETWORK LATENCY TEST TOOL")
print("=" * 60)

while True:
    try:
        TARGET_IP = str(ipaddress.ip_address(
            input("Enter destination IP: ").strip()
        ))
        break
    except ValueError:
        print("Invalid IP address. Please try again.")

while True:
    try:
        TARGET_PORT = int(input("Enter TCP port: "))
        if 1 <= TARGET_PORT <= 65535:
            break
        print("Port must be between 1 and 65535.")
    except ValueError:
        print("Please enter a valid port number.")


def get_positive_number(prompt, default, converter):
    while True:
        try:
            value = input(f"{prompt} [{default}]: ").strip()
            result = converter(value) if value else default

            if result > 0 and math.isfinite(result):
                return result

            print("Value must be greater than zero.")
        except (ValueError, OverflowError):
            print("Please enter a valid positive number.")


TEST_COUNT = get_positive_number(
    "Enter number of tests", 1000, int
)

INTERVAL_SECONDS = get_positive_number(
    "Enter interval in seconds", 1, float
)

TIMEOUT_SECONDS = get_positive_number(
    "Enter timeout in seconds", 2, float
)

# ==========================================
# OUTPUT FILES
# ==========================================

OUTPUT_DIR = Path.home() / "Desktop" / "TCP_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

RAW_FILE = OUTPUT_DIR / f"tcp_raw_{timestamp}.csv"
SUMMARY_FILE = OUTPUT_DIR / f"tcp_summary_{timestamp}.csv"

# ==========================================
# PERCENTILE FUNCTION
# ==========================================

def percentile(values, percent):
    if not values:
        return None

    values = sorted(values)

    rank = (len(values) - 1) * percent / 100

    lower = math.floor(rank)
    upper = math.ceil(rank)

    result = values[lower] + (
        values[upper] - values[lower]
    ) * (rank - lower)

    return round(result, 2)


# ==========================================
# TCP CONNECTION TEST
# ==========================================

def tcp_ping(ip, port, timeout):

    start = time.perf_counter()

    try:
        with socket.create_connection(
            (ip, port), timeout=timeout
        ):
            latency_ms = (
                time.perf_counter() - start
            ) * 1000

            return "Success", round(latency_ms, 3), "Connected"

    except socket.timeout:
        return "Timeout", None, "Connection timeout"

    except ConnectionRefusedError:
        return "Refused", None, "Connection refused"

    except OSError as error:
        return "Error", None, str(error)


# ==========================================
# MAIN TCP TEST
# ==========================================

def main():

    results = []
    latencies = []

    print("\n" + "=" * 60)
    print("           WINDOWS TCP PING TEST")
    print("=" * 60)
    print(f"Destination : {TARGET_IP}")
    print(f"TCP Port    : {TARGET_PORT}")
    print(f"Test Count  : {TEST_COUNT}")
    print(f"Interval    : {INTERVAL_SECONDS} seconds")
    print(f"Timeout     : {TIMEOUT_SECONDS} seconds")
    print("=" * 60)

    test_start = time.monotonic()

    try:
        for sequence in range(1, TEST_COUNT + 1):

            probe_start = time.monotonic()

            timestamp_now = datetime.now().isoformat(
                timespec="milliseconds"
            )

            status, latency, note = tcp_ping(
                TARGET_IP,
                TARGET_PORT,
                TIMEOUT_SECONDS
            )

            results.append({
                "Timestamp": timestamp_now,
                "Sequence": sequence,
                "Destination": TARGET_IP,
                "Port": TARGET_PORT,
                "Status": status,
                "TCP_Connect_ms": latency,
                "Note": note
            })

            if status == "Success":
                latencies.append(latency)

            print(
                f"Seq={sequence} | "
                f"TCP={TARGET_PORT} | "
                f"Status={status} | "
                f"Connect={latency} ms | "
                f"{note}"
            )

            elapsed = time.monotonic() - probe_start
            remaining = INTERVAL_SECONDS - elapsed

            if remaining > 0 and sequence < TEST_COUNT:
                time.sleep(remaining)

    except KeyboardInterrupt:
        print("\nTest interrupted. Saving collected results...")

    duration = time.monotonic() - test_start

    # ==========================================
    # TCP STATISTICS
    # ==========================================

    transmitted = len(results)
    successful = len(latencies)
    failed = transmitted - successful

    failure_percent = (
        failed / transmitted * 100
        if transmitted else 0
    )

    timeout_count = sum(
        1 for r in results if r["Status"] == "Timeout"
    )

    refused_count = sum(
        1 for r in results if r["Status"] == "Refused"
    )

    error_count = sum(
        1 for r in results if r["Status"] == "Error"
    )

    current_streak = 0
    max_failure_streak = 0

    for r in results:
        if r["Status"] != "Success":
            current_streak += 1
            max_failure_streak = max(
                max_failure_streak,
                current_streak
            )
        else:
            current_streak = 0

    summary = {
        "Destination": TARGET_IP,
        "Port": TARGET_PORT,
        "Attempts": transmitted,
        "Successful": successful,
        "Failed": failed,
        "Failure_Percent": round(failure_percent, 4),
        "Timeouts": timeout_count,
        "Refused": refused_count,
        "Errors": error_count,
        "Max_Consecutive_Failures": max_failure_streak,
        "Duration_Seconds": round(duration, 2),
        "Min_TCP_ms": min(latencies) if latencies else None,
        "Avg_TCP_ms": (
            round(statistics.mean(latencies), 2)
            if latencies else None
        ),
        "Max_TCP_ms": max(latencies) if latencies else None,
        "P10_ms": percentile(latencies, 10),
        "P50_ms": percentile(latencies, 50),
        "P95_ms": percentile(latencies, 95),
        "P99_ms": percentile(latencies, 99)
    }

    # ==========================================
    # EXPORT RAW RESULTS
    # ==========================================

    if results:
        with open(RAW_FILE, "w", newline="") as file:
            writer = csv.DictWriter(
                file,
                fieldnames=results[0].keys()
            )
            writer.writeheader()
            writer.writerows(results)

    # ==========================================
    # EXPORT SUMMARY
    # ==========================================

    with open(SUMMARY_FILE, "w", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=summary.keys()
        )
        writer.writeheader()
        writer.writerow(summary)

    # ==========================================
    # DISPLAY RESULTS
    # ==========================================

    print("\n" + "=" * 60)
    print("             TCP TEST SUMMARY")
    print("=" * 60)

    for key, value in summary.items():
        print(f"{key:30}: {value}")

    if results:
        print("\nRaw CSV:", RAW_FILE)

    print("Summary CSV:", SUMMARY_FILE)


if __name__ == "__main__":
    main()


        TCP NETWORK LATENCY TEST TOOL

           WINDOWS TCP PING TEST
Destination : 1.1.1.1
TCP Port    : 443
Test Count  : 10
Interval    : 2.0 seconds
Timeout     : 2.0 seconds
Seq=1 | TCP=443 | Status=Success | Connect=4.737 ms | Connected
Seq=2 | TCP=443 | Status=Success | Connect=4.367 ms | Connected
Seq=3 | TCP=443 | Status=Success | Connect=22.414 ms | Connected
Seq=4 | TCP=443 | Status=Success | Connect=4.685 ms | Connected
Seq=5 | TCP=443 | Status=Success | Connect=3.645 ms | Connected
Seq=6 | TCP=443 | Status=Success | Connect=3.838 ms | Connected
Seq=7 | TCP=443 | Status=Success | Connect=4.657 ms | Connected
Seq=8 | TCP=443 | Status=Success | Connect=9.313 ms | Connected
Seq=9 | TCP=443 | Status=Success | Connect=3.913 ms | Connected
Seq=10 | TCP=443 | Status=Success | Connect=3.86 ms | Connected

             TCP TEST SUMMARY
Destination                   : 1.1.1.1
Port                          : 443
Attempts                      : 10
Successful                    : 10
F

In [1]:
# Additional protocol lab: ICMP, UDP, TCP, DNS, HTTPS, and routing.
# The notebook already has ICMP and TCP connection tests. These examples
# use loopback for ICMP/UDP/TCP; DNS, HTTPS, and traceroute need network access.
# Other useful topics: MTU/path-MTU discovery, ARP (local LAN), IPv6 routing,
# packet capture, and the difference between reachability and service health.

# Windows ICMP Echo tests against local IPv4 and IPv6 stacks (no external host).
import subprocess

for family, address in (("IPv4", "127.0.0.1"), ("IPv6", "::1")):
    command = ["ping", "-4" if family == "IPv4" else "-6", "-n", "2", "-w", "1000", address]
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=5, check=False)
        print(f"{family} loopback ({address}) exit={result.returncode}")
        print(result.stdout.strip() or result.stderr.strip())
    except subprocess.TimeoutExpired:
        print(f"{family} loopback ping process exceeded 5 seconds")


IPv4 loopback (127.0.0.1) exit=0
Pinging 127.0.0.1 with 32 bytes of data:
Reply from 127.0.0.1: bytes=32 time<1ms TTL=128
Reply from 127.0.0.1: bytes=32 time<1ms TTL=128

Ping statistics for 127.0.0.1:
    Packets: Sent = 2, Received = 2, Lost = 0 (0% loss),
Approximate round trip times in milli-seconds:
    Minimum = 0ms, Maximum = 0ms, Average = 0ms
IPv6 loopback (::1) exit=0
Pinging ::1 with 32 bytes of data:
Reply from ::1: time<1ms 
Reply from ::1: time<1ms 

Ping statistics for ::1:
    Packets: Sent = 2, Received = 2, Lost = 0 (0% loss),
Approximate round trip times in milli-seconds:
    Minimum = 0ms, Maximum = 0ms, Average = 0ms


In [2]:
# UDP has no connection handshake or delivery guarantee. This local echo demo
# supplies an application response so the client can measure a round trip.
import socket
import threading
import time

server = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
server.bind(("127.0.0.1", 0))
server.settimeout(3)
server_address = server.getsockname()

def reply_once():
    message, client_address = server.recvfrom(1024)
    server.sendto(message, client_address)

server_thread = threading.Thread(target=reply_once, daemon=True)
server_thread.start()
payload = b"UDP loopback echo"
with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as client:
    client.settimeout(3)
    started = time.perf_counter()
    client.sendto(payload, server_address)
    response, _ = client.recvfrom(1024)
    round_trip_ms = (time.perf_counter() - started) * 1000

server_thread.join(timeout=3)
server.close()
assert response == payload, "UDP echo payload did not match"
print(f"UDP echo received {response!r}; round trip={round_trip_ms:.3f} ms")


UDP echo received b'UDP loopback echo'; round trip=0.578 ms


In [3]:
# TCP loopback echo: demonstrates connection setup plus application data.
# This is a local teaching test; it does not contact or scan a remote service.
import socket
import threading
import time

listener = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
listener.bind(("127.0.0.1", 0))
listener.listen(1)
listener.settimeout(5)
server_address = listener.getsockname()

def serve_one_client():
    connection, _ = listener.accept()
    with connection:
        data = connection.recv(1024)
        connection.sendall(data)

server_thread = threading.Thread(target=serve_one_client, daemon=True)
server_thread.start()
payload = b"TCP loopback echo"
started = time.perf_counter()
with socket.create_connection(server_address, timeout=3) as client:
    connect_ms = (time.perf_counter() - started) * 1000
    client.sendall(payload)
    client.shutdown(socket.SHUT_WR)
    response = client.recv(1024)

server_thread.join(timeout=5)
listener.close()
assert response == payload, "TCP echo payload did not match"
print(f"TCP connected in {connect_ms:.3f} ms; echoed {response!r}")
print("A successful TCP connect proves a port accepted the handshake, not that every application request will succeed.")


TCP connected in 0.747 ms; echoed b'TCP loopback echo'
A successful TCP connect proves a port accepted the handshake, not that every application request will succeed.


In [4]:
# DNS resolution and one HTTPS request (requires internet/DNS access).
# HTTPS uses TLS; urllib validates the server certificate by default.
import socket
from urllib.request import urlopen

host = "example.com"
addresses = sorted({
    result[4][0]
    for result in socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
})
print(f"DNS answers for {host}: {addresses}")

with urlopen(f"https://{host}/", timeout=5) as response:
    print(f"HTTPS status={response.status}; content-type={response.headers.get('Content-Type')}")
    print(f"Final URL: {response.url}")


DNS answers for example.com: ['104.20.23.154', '172.66.147.243']
HTTPS status=200; content-type=text/html; charset=utf-8
Final URL: https://example.com/


In [5]:
# Optional Windows route-path diagnostic; sends probes toward a public IP.
# Bounded to 6 hops and 500 ms per probe. Routers/firewalls may hide hops.
import subprocess

result = subprocess.run(
    ["tracert", "-4", "-d", "-h", "6", "-w", "500", "1.1.1.1"],
    capture_output=True,
    text=True,
    timeout=12,
    check=False,
)
print(result.stdout or result.stderr)
print(f"tracert exit code: {result.returncode}")

# Also study MTU/path-MTU discovery, ARP for same-LAN address resolution,
# and packet capture for packet-level evidence.



Tracing route to 1.1.1.1 over a maximum of 6 hops

  1     2 ms     2 ms     1 ms  10.0.254.23 
  2     *        *        *     Request timed out.
  3     *        *        *     Request timed out.
  4     *        *        *     Request timed out.
  5     *        *        *     Request timed out.
  6     *        *        *     Request timed out.

Trace complete.

tracert exit code: 0
